# Type/Token Ratio and Its Trap

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* Vocabulary richness; length dependence and standardised TTR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02c_type_token_ratio.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Type/token ratio is the classic measure of vocabulary richness: distinct words
divided by total words. A text that never repeats itself scores 1.0.

It is easy to compute, widely reported, and **wrong in almost every published use**
— because it falls as a text gets longer, no matter how rich the vocabulary is.
This notebook shows you why, and what to do instead.

## The tooling

**`collections.Counter`** is the whole engine of frequency analysis — a
standard-library dictionary that counts. **matplotlib** draws the distributions.
The field's dedicated tools (**AntConc**, **#LancsBox**) wrap the same counting
in a GUI.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
def ttr(text):
    toks = tokenise(text)
    return len(set(toks)) / len(toks) if toks else 0.0

work = df[df["language"] == "English"].copy()
work["ttr"] = work["text"].map(ttr)
print(work[["category", "n_words", "ttr"]].head(8).to_string(index=False))
print()
print(f"mean TTR: {work['ttr'].mean():.3f}")

## The trap, demonstrated

Take one document and compute its TTR at 100 words, 200, 400, and so on. The
vocabulary is the same author, same article, same richness — but the number
falls steadily.

In [ ]:
doc = work.sort_values("n_words", ascending=False).iloc[0]
toks = tokenise(doc["text"])
sizes = [n for n in range(100, min(len(toks), 2000) + 1, 100)]
vals = [len(set(toks[:n])) / n for n in sizes]

for n, v in list(zip(sizes, vals))[:6]:
    print(f"  first {n:>5} tokens: TTR = {v:.3f}")
print("  …")
print(f"  first {sizes[-1]:>5} tokens: TTR = {vals[-1]:.3f}")
print()
print("Same document, same vocabulary, falling number. TTR measures length")
print("at least as much as it measures richness.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sizes, vals, marker="o")
ax.set_xlabel("tokens read"); ax.set_ylabel("type/token ratio")
ax.set_title("TTR falls with length — in a single unchanging document")
ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## What that means for comparisons

If you compare TTR between two groups of documents and one group is longer, you
will find a difference. It will be real, statistically significant, and about
length.

In [ ]:
# Watch the artefact appear in a real comparison.
grp = work.groupby("category").agg(docs=("ttr", "size"),
                                   mean_ttr=("ttr", "mean"),
                                   median_words=("n_words", "median"))
grp = grp[grp["docs"] >= 15].sort_values("mean_ttr", ascending=False)
print(grp.to_string(float_format=lambda v: f"{v:.3f}"))
print()
r = np.corrcoef(work["n_words"], work["ttr"])[0, 1]
print(f"correlation between document length and TTR: {r:+.3f}")
print("Strongly negative. The category ranking above largely reproduces the")
print("length ranking — so it is not evidence about vocabulary at all.")

## The fix: standardised TTR

Compute TTR over fixed-size windows and average them. Every document is then
measured on the same amount of text, so length cannot drive the result.

Documents shorter than the window are excluded — and saying how many you dropped
is part of reporting the measure honestly.

In [ ]:
WINDOW = 200                # ← change this

def sttr(text, window=WINDOW):
    # Mean TTR over consecutive fixed-size windows; None if the text is too short.
    toks = tokenise(text)
    if len(toks) < window:
        return None
    chunks = [toks[i:i + window] for i in range(0, len(toks) - window + 1, window)]
    return float(np.mean([len(set(c)) / window for c in chunks]))

work["sttr"] = work["text"].map(sttr)
usable = work.dropna(subset=["sttr"])
print(f"{len(usable):,} of {len(work):,} documents are at least {WINDOW} tokens long")
print()
r2 = np.corrcoef(usable["n_words"], usable["sttr"])[0, 1]
print(f"correlation between length and STTR: {r2:+.3f}   (TTR was {r:+.3f})")
print()
print(usable.groupby("category").agg(docs=("sttr", "size"), mean_sttr=("sttr", "mean"))
      .query("docs >= 15").sort_values("mean_sttr", ascending=False)
      .to_string(float_format=lambda v: f"{v:.3f}"))

### 🔧 Your turn 1

Set `WINDOW` to 100, then 500, and re-run.

A smaller window keeps more documents but measures less text each; a larger one
is more stable but throws documents away. Which trade-off would you defend, and
how many documents did you lose?

### 🔧 Your turn 2

Run the same comparison across `language` instead of `category`.

Before you interpret any difference: these are different languages with different
morphology, so a language that marks grammar with affixes will show more types
per token *by construction*. Is that a finding about vocabulary richness, or
about morphology?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The correlation with length should collapse towards zero — that
is the whole point. At `WINDOW = 100` you keep nearly every document but each
measurement is noisy; at 500 you may lose half the corpus, and the documents you
lose are systematically the short ones, which is its own bias. 200 is a common
compromise. Report the window and the number of documents excluded.

**Your turn 2.** It is about morphology, and it is the single commonest
misreading of TTR in cross-linguistic work. Hausa, Swahili and Yoruba encode
grammatical relations differently, so a "richer" vocabulary may just be more
inflected forms of the same roots. Without a lemmatiser — and none exists for
most African languages — cross-language TTR is not interpretable as richness.
Within one language, comparing genres or authors, it is fine.

</details>

## Where this stops

You can measure vocabulary richness without accidentally measuring length. Next:
**02d — document length itself**, which turns out to be a real variable rather
than only a nuisance.